# Document RAG System.

In [ ]:
# install needed libraries to use
!pip -q install langchain langchain-google-genai langchain-community google-genai faiss-cpu tiktoken python-dotenv pypdf langchain-huggingface sentence-transformers

In [18]:
# import libraries
import os
from google.colab import userdata

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# set environment variables for google and huggingface

os.environ['GOOGLE_API_KEY'] = userdata.get("GOOGLE_API_KEY")
os.environ['HUGGINGFACEHUB_ACCESS_TOKEN'] = userdata.get("HUGGINGFACEHUB_ACCESS_TOKEN")

### Load keys.

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

openai_key = os.getenv("OPENAI_API_KEY")
gemini_key = os.getenv("GEMINI_API_KEY")

print("OpenAI key loaded:", bool(openai_key))
# print("OpenAI key:", openai_key)

print("\nGemini key loaded:", bool(gemini_key))
# print("Gemini key:", gemini_key)


OpenAI key loaded: True

Gemini key loaded: True


In [ ]:
# import necessary libraries

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI,GoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings

### Test key with prompt.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=gemini_key
)

response = llm.invoke("Hello")
print(response)


E0000 00:00:1759185374.177116  102820 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


content='Hello! How can I help you today?' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []} id='run--95cf3202-9c6a-49d9-a562-0aa5e92f20c8-0' usage_metadata={'input_tokens': 2, 'output_tokens': 32, 'total_tokens': 34, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 23}}
Hello! How can I help you today?


In [ ]:
print(response.content)

## Load document.

In [ ]:
# load and read PDF file

loader = PyPDFLoader("ChenZhang_cropmapping_ReviewPaper.pdf")
docs = loader.load()

In [66]:
len(docs)

8

In [67]:
# Front cover of PDF
print(docs[0].page_content) 

THE FEDERAL UNIVERSITY OF
TECHNOLOGY, AKURE
URP 303 TERM PAPER ON THE TOPIC:
VARIOUS LANDFORMS AND
THEIR MANAGEMENT
BY
NAME: LADE-IGE TIMILEHIN
MATRIC NO: RSG/22/9701
LECTURER: DR. J.A OLANIBI


In [68]:
# Back cover page
print(docs[1].page_content)

OUTLINE
I. Introduction
A. Definition of Landform
B. Importance of Landform Management
II. Types of Landform
A. Mountains
1. Characteristics
2. Management Strategies
B. Plateaus
1. Characteristics
2. Management Strategies
C. Valleys
1. Characteristics
2. Management Strategies
D. Plains
1. Characteristics
2. Management Strategies
E. Deserts
1. Characteristics
2. Management Strategies
III. Landform Management
A. Sustainable Land Use Practices
B. Soil Conservation Techniques
C. Water Resource Management
D. Biodiversity Conservation
IV. Future Directions in Landform Management
A. Climate Change and Landform
B. Technological Advancements in Management
C. Community Involvement and Stakeholder Engagement
V. Conclusion
A. Recap of Key Points


In [69]:
# first page after cover
print(docs[2].page_content)

I. Introduction
A. Definition of Landform
Landforms are natural physical features of the Earth's surface, shaped by
various geological processes over time. They include mountains, plateaus,
valleys, plains, and deserts, each exhibiting distinct characteristics formed
through processes like erosion, sedimentation, and tectonic activity.
B. Importance of Landform Management
1. Ecosystem Preservation: Proper landform management helps
maintain ecological balance by protecting habitats and biodiversity. Healthy
ecosystems contribute to clean air and water, soil fertility, and the overall
health of the environment.
2. Disaster Risk Reduction: Effective management of landform can
mitigate natural hazards such as landslides, floods, and erosion.
Implementing strategies like slope stabilization, flood control measures, and
sustainable land use practices helps protect lives and property.
3. Sustainable Resource Management: Landform provide vital
resources such as minerals, water, and arable land

In [71]:
# second page after cover
print(docs[3].page_content)

infrastructure projects take into account natural landform, reducing
environmental impact and improving quality of life for residents.
Landform management is crucial for maintaining ecological balance,
supporting biodiversity, and ensuring sustainable land use. Effective
management practices help mitigate natural hazards, preserve natural
resources, and enhance the livelihoods of communities dependent on
these landform. Moreover, sound management strategies can
counteract the adverse effects of urbanization, agriculture, and
climate change.
II. Types of Landform
A. Mountains: Mountains are elevated landform characterized by steep
slopes and varying altitudes. They often contain unique ecosystems and
climate zones due to their elevation, leading to distinct flora and fauna. They
are created by tectonic forces, volcanic activity or erosion.
Characteristics:
 Cooler temperatures as altitude increases
 Found in ranges (e.g., Himalayas, Andes, Alps)
 Support diverse ecosystems based on a

## Split texts.

In [72]:
# split into chunks

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = splitter.split_documents(docs)

In [73]:
len(chunks)

15

In [76]:
print(chunks[4].page_content)

and reduce the impacts of extreme weather events.
6. Cultural and Recreational Value: Many landforms have cultural
significance and provide opportunities for recreation and tourism. Proper
management enhances their aesthetic and cultural value, attracting visitors
and supporting local economies.
7. Community Engagement and Resilience: Involving local
communities in landform management fosters a sense of ownership and
responsibility. This engagement can lead to more effective and culturally
relevant management strategies, enhancing community resilience.
8. Urban Planning and Development: Effective landform
management supports sustainable urban development by ensuring that


In [77]:
embeds = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeds)

## Retrieval.

In [81]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [82]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7d4f78937860>, search_kwargs={'k': 5})

In [83]:
retriever.invoke("what is the main topic of the document?")

[Document(id='340fea89-7484-42db-9332-1a5384d91e15', metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2025-03-24T10:55:33+01:00', 'author': 'USER', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2025-03-24T10:55:33+01:00', 'sourcemodified': "D:20250324105533+01'00'", 'subject': '', 'title': '', 'trapped': '/False', 'source': 'URP TERM PAPER.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content='THE FEDERAL UNIVERSITY OF\nTECHNOLOGY, AKURE\nURP 303 TERM PAPER ON THE TOPIC:\nVARIOUS LANDFORMS AND\nTHEIR MANAGEMENT\nBY\nNAME: LADE-IGE TIMILEHIN\nMATRIC NO: RSG/22/9701\nLECTURER: DR. J.A OLANIBI'),
 Document(id='d3a78201-0d44-47e2-9db6-1eb4cb0daefa', metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2025-03-24T10:55:33+01:00', 'author': 'USER', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2025-03-24T10:55:33+01:00', 'sourcemodified': "D:20250324105533+01'00'", 'subject': '', 'title': '', 'trapped': '/False', 'sour

## Augmentation.

In [84]:
llm_gen = GoogleGenerativeAI(model="models/gemini-1.5-flash")

E0000 00:00:1759060425.855264   47311 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [86]:
prompt = PromptTemplate(
    template = """
    You are a helpful assistant.
    Answer ONLY from the provided transcript context.
    If the context IS INSUFFICIENT, just say you don't know and probably need more information.

    {context}

    Question: {question}
    """,
    input_variables=["context","question"]
)

In [89]:
question = "Is the context of stars is mentioned in this document? If yes, then what was discussed?"
retrieved_docs = retriever.invoke(question)

In [90]:
retrieved_docs

[Document(id='0da02404-b6cf-460b-b701-8d9563f2c104', metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2025-03-24T10:55:33+01:00', 'author': 'USER', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2025-03-24T10:55:33+01:00', 'sourcemodified': "D:20250324105533+01'00'", 'subject': '', 'title': '', 'trapped': '/False', 'source': 'URP TERM PAPER.pdf', 'total_pages': 8, 'page': 6, 'page_label': '7'}, page_content='and promotes sustainable practices. Engaging stakeholders ensures that\nmanagement strategies are culturally sensitive and economically viable.\nV. Conclusion\nA. Recap of Key Points\nUnderstanding the characteristics and management strategies associated\nwith various landform is vital.Identifying the geological processes that'),
 Document(id='340fea89-7484-42db-9332-1a5384d91e15', metadata={'producer': '', 'creator': 'WPS Writer', 'creationdate': '2025-03-24T10:55:33+01:00', 'author': 'USER', 'comments': '', 'company': '', 'keywords': '', 'moddate': 

In [91]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [92]:
context_text

'and promotes sustainable practices. Engaging stakeholders ensures that\nmanagement strategies are culturally sensitive and economically viable.\nV. Conclusion\nA. Recap of Key Points\nUnderstanding the characteristics and management strategies associated\nwith various landform is vital.Identifying the geological processes that\n\nTHE FEDERAL UNIVERSITY OF\nTECHNOLOGY, AKURE\nURP 303 TERM PAPER ON THE TOPIC:\nVARIOUS LANDFORMS AND\nTHEIR MANAGEMENT\nBY\nNAME: LADE-IGE TIMILEHIN\nMATRIC NO: RSG/22/9701\nLECTURER: DR. J.A OLANIBI\n\n\uf0b7 Limited vegetation adapted to dry conditions\n\uf0b7 Large temperature variations between day and night\n\uf0b7 Examples: Sahara Desert (Africa), Atacama Desert (Chile), Thar Desert\n(India)\n\uf0b7\nManagement Strategies: Management of deserts focuses on water\nconservation techniques, such as rainwater harvesting and sustainable\nland use planning to prevent habitat degradation and promote\nbiodiversity.\n\nand reduce the impacts of extreme weather e

In [93]:
final_prompt = prompt.invoke({"context":context_text,"question":question})

In [94]:
final_prompt

StringPromptValue(text="\n    You are a helpful assistant.\n    Answer ONLY from the provided transcript context.\n    If the context IS INSUFFICIENT, just say you don't know and probably need more information.\n\n    and promotes sustainable practices. Engaging stakeholders ensures that\nmanagement strategies are culturally sensitive and economically viable.\nV. Conclusion\nA. Recap of Key Points\nUnderstanding the characteristics and management strategies associated\nwith various landform is vital.Identifying the geological processes that\n\nTHE FEDERAL UNIVERSITY OF\nTECHNOLOGY, AKURE\nURP 303 TERM PAPER ON THE TOPIC:\nVARIOUS LANDFORMS AND\nTHEIR MANAGEMENT\nBY\nNAME: LADE-IGE TIMILEHIN\nMATRIC NO: RSG/22/9701\nLECTURER: DR. J.A OLANIBI\n\n\uf0b7 Limited vegetation adapted to dry conditions\n\uf0b7 Large temperature variations between day and night\n\uf0b7 Examples: Sahara Desert (Africa), Atacama Desert (Chile), Thar Desert\n(India)\n\uf0b7\nManagement Strategies: Management of de

## Answer Generation.

In [ ]:
response = llm.invoke(final_prompt)

In [ ]:
response.content

"I don't know and probably need more information. The context of stars is not mentioned in this document."

## Build chain.

In [102]:
# import libraries for chain building
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def reformat_doc(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [106]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(reformat_doc),
    'question': RunnablePassthrough()
}
)

In [ ]:
parallel_chain.invoke('what are the future directions in landform management')

{'context': "OUTLINE\nI. Introduction\nA. Definition of Landform\nB. Importance of Landform Management\nII. Types of Landform\nA. Mountains\n1. Characteristics\n2. Management Strategies\nB. Plateaus\n1. Characteristics\n2. Management Strategies\nC. Valleys\n1. Characteristics\n2. Management Strategies\nD. Plains\n1. Characteristics\n2. Management Strategies\nE. Deserts\n1. Characteristics\n2. Management Strategies\nIII. Landform Management\nA. Sustainable Land Use Practices\nB. Soil Conservation Techniques\nC. Water Resource Management\nD. Biodiversity Conservation\nIV. Future Directions in Landform Management\nA. Climate Change and Landform\nB. Technological Advancements in Management\nC. Community Involvement and Stakeholder Engagement\nV. Conclusion\nA. Recap of Key Points\n\nTHE FEDERAL UNIVERSITY OF\nTECHNOLOGY, AKURE\nURP 303 TERM PAPER ON THE TOPIC:\nVARIOUS LANDFORMS AND\nTHEIR MANAGEMENT\nBY\nNAME: LADE-IGE TIMILEHIN\nMATRIC NO: RSG/22/9701\nLECTURER: DR. J.A OLANIBI\n\nI. Int

In [ ]:
parse = StrOutputParser()

In [109]:
main_chain = parallel_chain | prompt | llm | parse

In [113]:
print(main_chain.invoke("what are the future directions in landform management"))

Based on the provided transcript context, the future directions in landform management include:

*   **Technological Advancements in Management:** This involves incorporating technology and remote sensing techniques to enhance the study and management of diverse landforms.
*   **Community Involvement and Stakeholder Engagement:** Collaborating with local communities and stakeholders is key to developing effective landform conservation strategies.
*   The text also mentions conducting regular assessments and monitoring of landforms to detect any changes or threats early on.

While "Climate Change and Landform" is listed as a future direction in the outline, the provided text does not offer specific details or strategies related to it. Therefore, the context is insufficient to elaborate on this point.


In [114]:
print(main_chain.invoke("what are the future directions in landform management. List and explain them in one sentence each."))

I don't know and probably need more information. The provided text lists the future directions but does not explain them.
